# AES-128 Algorithm w/ random fault emulation

In [266]:
import numpy as np
from base64 import b64encode, b64decode
import random
import converter
from converter import *
import importlib
from Crypto.Util.Padding import pad, unpad
from collections import Counter

importlib.reload(converter)

class AES:
    def __init__(self, should_fault=False):
        self.ROUND = 10
        self.ORDER = 4
        self.ROUNDKEY = []
        self.should_fault = should_fault
        self.did_fault = False
        # self.random_round = random.randint(self.ROUND-2, self.ROUND+1)
        self.random_round = self.ROUND - 1
        self.random_step = random.randint(1, 5)
        print(f"random_round: {self.random_round}, random_step: {self.random_step}")
    
    # Key Scheduling
    def __keySchedule(self, KEY):
        hexKey = keyToHexArray(KEY)
        self.ROUNDKEY.append(hexKey)
        for i in range(0, self.ROUND):
            prev_arr = self.ROUNDKEY[-1]
            last_col = prev_arr[self.ORDER-1]
            shift_col = arrayShift(last_col)
            sbox_col = arraySbox(shift_col)
            col_1 = xorArray(prev_arr[0], sbox_col, rcon=i)
            col_2 = xorArray(col_1, prev_arr[1])
            col_3 = xorArray(col_2, prev_arr[2])
            col_4 = xorArray(col_3, prev_arr[3])
            new_arr = np.array([col_1, col_2, col_3, col_4])
            self.ROUNDKEY.append(new_arr)

    def __stateFault(self, state, round_num, step):
        if self.did_fault or not self.should_fault: 
            return state
        if (round_num == self.ROUND - 1 and step == 3) or (round_num == self.random_round and self.random_step == step):   
        # if round_num == self.ROUND -1 and step == 2:
        # if round_num == self.ROUND -1 and step == self.random_step:
            self.did_fault = True

            random_flat_index = np.random.default_rng().integers(0, state.size)
            row, col = np.unravel_index(random_flat_index, state.shape)
            
            old_state = state.copy()
            state[row, col] ^= random.randint(1, 0xFF)
            
            print("FAULT INSERTED")
            print(f"\tRound Number: {round_num}")
            print(f"\tStep: {step}")
            print(f"\tFlat index: {random_flat_index}")
            print(f"\tOld state: {old_state}")
            print(f"\tNew state: {state}")
            
            return np.array(state)
        else:
            return state

    # Encryption Process
    def __encryptProcess(self, TEXT):
        hexData = keyToHexArray(TEXT)
        cipher_arr = addRoundKey(hexData, self.ROUNDKEY[0])
        for i in range(1, self.ROUND+1):
            arr = cipher_arr
            arr = self.__stateFault(arr, i, 1)
            arr = subBytes(arr)
            arr = self.__stateFault(arr, i, 2)
            arr = shiftRow(arr)
            if(i != self.ROUND):
                arr = self.__stateFault(arr, i, 3)
                arr = mixColumn(arr)
                arr = self.__stateFault(arr, i, 4)
            arr = addRoundKey(arr, self.ROUNDKEY[i])
            cipher_arr = arr
        return cipher_arr

    # Encryption Add Padding
    def __addPadding(self, data):
        bytes = 16
        bits_arr = []
        while(True):
            if(len(data) > bytes):
                bits_arr.append(data[:bytes])
                data = data[bytes:]
            else:
                space = bytes-len(data)
                bits_arr.append(data + chr(space).encode('latin-1')*space)
                break
        return bits_arr

    # Decryption Process
    def __decryptProcess(self, CIPHER_HEX):
        hexData = hexToMatrix(CIPHER_HEX)
        plain_arr = addRoundKey(hexData, self.ROUNDKEY[-1])
        for i in range(self.ROUND-1, -1, -1):
            arr = plain_arr
            arr = shiftRow(arr, left=False)
            arr = subBytes(arr, inverse=True)
            arr = addRoundKey(arr, self.ROUNDKEY[i])
            if(i != 0):
                arr = inverseMixColumn(arr)
            plain_arr = arr
        return plain_arr

    # Decryption Delete Padding
    def __delPadding(self, data):
        verify = data[-1]
        if(verify >= 1 and verify <= 15):
            pad = data[16-verify:]
            sameCount = pad.count(verify)
            if(sameCount == verify):
                return data[:16-verify]
            return data
        return data

    #Encryption
    def encrypt(self, KEY, TEXT, type='hex'):
        text_arr = self.__addPadding(TEXT)
        self.__keySchedule(KEY)
        hex_ecrypt=''
        for i in text_arr:
            cipher_matrix = self.__encryptProcess(i)
            cipher_text = list(np.array(cipher_matrix, order='C').reshape(-1,))
            for j in cipher_text:
                hex_ecrypt+=f'{j:02x}'
        self.ROUNDKEY = []
        #conversion
        if(type == 'b64'):
            return b64encode(bytes.fromhex(hex_ecrypt)).decode()
        if(type == '0b'):
            return f'{int(hex_ecrypt, 16):0>b}'
        if(type == '__all__'):
            return {
                'hex': hex_ecrypt,
                'b64': b64encode(bytes.fromhex(hex_ecrypt)).decode(),
                '0b': bin(int(hex_ecrypt, 16))[2:].zfill(len(hex_ecrypt) * 4)
            }
        return hex_ecrypt

    # Decryption
    def decrypt(self, KEY, CIPHER, type='hex'):
        if type in ['hex', '0b', 'b64']:
            self.__keySchedule(KEY)
            data = ''

            if(type == 'b64'):
                CIPHER = b64decode(CIPHER).hex()

            if(type == '0b'):
                CIPHER = hex(int(CIPHER, 2)).replace('0x','')

            if(len(CIPHER) % 32 == 0 and len(CIPHER) > 0):
                examine = CIPHER
                while(len(examine) != 0):
                    plain_matrix = self.__decryptProcess(examine[:32])
                    plain_arr = list(np.array(plain_matrix, order='C').reshape(-1,))
                    plain_arr = self.__delPadding(plain_arr)
                    for j in plain_arr:
                        data+=chr(j)
                    if(len(examine)==32):
                        examine=''
                    else:
                        examine=examine[32:]
                self.ROUNDKEY = []
                return data

            else:
                raise Exception(f"Hex: {CIPHER}, should be non-empty multiple of 32bits")

        else:
            raise Exception(f"type := ['hex', '0b', 'b64'] but got '{type}'")


# Key selection + encoding

In [267]:
key = "eecs475 eecs475s"
encoded_key = key.encode('latin-1')
print(f"Key: {key} Len: {len(key)}\nEncoded key: {encoded_key}\nHex: {encoded_key.hex()}")

Key: eecs475 eecs475s Len: 16
Encoded key: b'eecs475 eecs475s'
Hex: 65656373343735206565637334373573


# AES Verification

In [268]:
from Crypto.Cipher import AES as GoldAES
import binascii

def verify_aes(key_hex, plaintext_hex, your_ciphertext_hex):
    # 1. CRITICAL: Convert hex strings to raw bytes
    # PyCryptodome's C-backend REQUIRES bytes/bytearrays
    key_bytes = bytes.fromhex(key_hex)
    plaintext_bytes = plaintext_hex
    
    # 2. Encrypt using PyCryptodome (Reference)
    cipher = GoldAES.new(key_bytes, GoldAES.MODE_ECB)
    reference_ciphertext_bytes = cipher.encrypt(plaintext_bytes)
    
    # Convert reference back to hex for a fair comparison with your string
    reference_hex = reference_ciphertext_bytes.hex()

    # 3. Comparison
    is_correct = (reference_hex == your_ciphertext_hex)
    
    print("--- AES Verification ---")
    print(f"Standard: {reference_hex}")
    print(f"Yours:    {your_ciphertext_hex}")
    print(f"Status:   {'✅ MATCH' if is_correct else '❌ MISMATCH'}")
    
    return is_correct

In [269]:
message = "test"
encoded_message = message.encode('latin-1')

print(f"Message: {message}, len: {len(message)}, mencoded: {encoded_message}")

aes128 = AES()
c = aes128.encrypt(encoded_key, encoded_message, 'hex')
p = aes128.decrypt(encoded_key, c)

print()
verify_aes(key.encode('latin-1').hex(), pad(encoded_message, 16), c)
print()

Message: test, len: 4, mencoded: b'test'
random_round: 9, random_step: 3

--- AES Verification ---
Standard: 94ca152c20ce867902d54b046c665de2
Yours:    94ca152c20ce867902d54b046c665de2
Status:   ✅ MATCH



# Faulty AES Singleshot Test

In [270]:
aes128_faulty = AES(should_fault=True)

fc = aes128_faulty.encrypt(encoded_key, encoded_message, 'hex')
fp = aes128_faulty.decrypt(encoded_key, fc)

normal_plaintext = p

print(f"Normal Ciphertext: {c}")
print(f"Faulty Ciphertext: {fc}")

print(f"Decrypted (validation) plaintext: {normal_plaintext}, equals original: {normal_plaintext == encoded_message}")

print(f"Faulty plaintext: {fp}")
print()
fcba = bytes.fromhex(fc)
cba = bytes.fromhex(c)

random_round: 9, random_step: 5
FAULT INSERTED
	Round Number: 9
	Step: 3
	Flat index: 3
	Old state: [[253  18 139 210]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
	New state: [[253  18 139  32]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
Normal Ciphertext: 94ca152c20ce867902d54b046c665de2
Faulty Ciphertext: 47ca152c20ce869a02d532046c5a5de2
Decrypted (validation) plaintext: test, equals original: False
Faulty plaintext: Ñß2HÍb&À¥Àj õ[



# Generate ciphertext/fault pairs

In [322]:
pairs=[]
aes128 = AES()

for i in range(20):
    aes128_faulty = AES(should_fault=True)
    
    c = aes128.encrypt(encoded_key, encoded_message, 'hex')
    fc = aes128_faulty.encrypt(encoded_key, encoded_message, 'hex')
    
    pairs.append((c, fc))

random_round: 9, random_step: 4
random_round: 9, random_step: 2
FAULT INSERTED
	Round Number: 9
	Step: 2
	Flat index: 4
	Old state: [[253 195 175 192]
 [107  18 173  53]
 [127  81 139 150]
 [ 72  13 230 210]]
	New state: [[253 195 175 192]
 [ 38  18 173  53]
 [127  81 139 150]
 [ 72  13 230 210]]
random_round: 9, random_step: 3
FAULT INSERTED
	Round Number: 9
	Step: 3
	Flat index: 1
	Old state: [[253  18 139 210]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
	New state: [[253   7 139 210]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
random_round: 9, random_step: 3
FAULT INSERTED
	Round Number: 9
	Step: 3
	Flat index: 5
	Old state: [[253  18 139 210]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
	New state: [[253  18 139 210]
 [107  47 230 192]
 [127  13 175  53]
 [ 72 195 173 150]]
random_round: 9, random_step: 5
FAULT INSERTED
	Round Number: 9
	Step: 3
	Flat index: 6
	Old state: [[253  18 139 210]
 [107  81 230 192]
 [127  13 175  53]
 [ 72 195 173

In [323]:
for i in pairs:
    print(f"{i[0]},{i[1]}")

94ca152c20ce867902d54b046c665de2,940d152cd8ce867902d54b376c6647e2
94ca152c20ce867902d54b046c665de2,d7ca152c20ce86dc02d57d046c1f5de2
94ca152c20ce867902d54b046c665de2,94ef152cedce867902d54b806c66a9e2
94ca152c20ce867902d54b046c665de2,946a152c3cce867902d54b6e6c66e9e2
94ca152c20ce867902d54b046c665de2,1bca152c20ce860302d526046c865de2
94ca152c20ce867902d54b046c665de2,94ca15ad20cecb7902744b044f665de2
94ca152c20ce867902d54b046c665de2,21ca152c20ce861a02d550046cd25de2
94ca152c20ce867902d54b046c665de2,74ca152c20ce86d802d5b4046c4e5de2
94ca152c20ce867902d54b046c665de2,94ca9c2c20cd867914d54b046c665d93
94ca152c20ce867902d54b046c665de2,5dca152c20ce867002d50d046c4b5de2
94ca152c20ce867902d54b046c665de2,9449152cf9ce867902d54bc06c66c9e2
94ca152c20ce867902d54b046c665de2,94ca622c2035867931d54b046c665d7c
94ca152c20ce867902d54b046c665de2,beca152c20ce866f02d596046ce85de2
94ca152c20ce867902d54b046c665de2,94ca15d520cef27902194b046f665de2
94ca152c20ce867902d54b046c665de2,3fca152c20ce861202d5d7046c625de2
94ca152c20

# AES DFA Solver

In [324]:
import time
import math
import itertools

class DFASolver:
    def __init__(self, pairs):
        self.pairs = pairs

    def recover_master_key_from_r10_key(self, round10_key_bytes):
        current_key = np.array(round10_key_bytes).reshape(4, 4)
        
        # forward schedule uses i from 0 to 9 to generate Rounds 1 through 10
        for i in range(9, -1, -1):
            prev_arr = np.zeros((4, 4), dtype=int)
            
            # Reverse the simple XORs for columns 3, 2, and 1
            prev_arr[3] = current_key[3] ^ current_key[2]
            prev_arr[2] = current_key[2] ^ current_key[1]
            prev_arr[1] = current_key[1] ^ current_key[0]
            
            shift_col = arrayShift(prev_arr[3])
            sbox_col = arraySbox(shift_col.copy())
            
            prev_arr[0] = xorArray(current_key[0], sbox_col, rcon=i)
            
            current_key = prev_arr
            
        master_key_hex = "".join([f"{b:02x}" for b in current_key.reshape(-1)])
        
        return master_key_hex

    def solve(self):
        mc_scalars = [
            [2, 1, 1, 3], # Fault was at Row 0
            [3, 2, 1, 1], # Fault was at Row 1
            [1, 3, 2, 1], # Fault was at Row 2
            [1, 1, 3, 2]  # Fault was at Row 3
        ]
        
        all_target_groups = [
            [0, 13, 10, 7],  # Cipheridx for Column 0
            [4, 1, 14, 11],  # Cipheridx for Column 1
            [8, 5, 2, 15],   # Cipheridx for Column 2
            [12, 9, 6, 3]    # Cipheridx for Column 3
        ]

        column_candidates = [None, None, None, None]

        start_time = time.time()
        
        for pair in self.pairs:
            normal_ciphertext = np.array(bytearray.fromhex(pair[0]), order='C').reshape(4,4)
            faulty_ciphertext = np.array(bytearray.fromhex(pair[1]), order='C').reshape(4,4)
            
            # Identify which column group this fault affects
            active_group_idx = -1
            for idx, group in enumerate(all_target_groups):
                if normal_ciphertext.flat[group[0]] != faulty_ciphertext.flat[group[0]]:
                    active_group_idx = idx
                    break

            #no differences
            if active_group_idx == -1:
                continue
                
            active_group = all_target_groups[active_group_idx]
            
            valid_tuples_for_pair = set()
            
            for row_scalars in mc_scalars: 
                for e in range(1, 256):    
                    
                    # Store valid key candidates for each of the 4 specific positions
                    byte_cands = [[], [], [], []]

                    # system of equations here!
                    for i in range(4): 
                        scalar = row_scalars[i]
                        ci = active_group[i] 
                        
                        c = normal_ciphertext.flat[ci]
                        fc = faulty_ciphertext.flat[ci]
                        
                        for k_guess in range(256):
                            if InvSbox(c ^ k_guess) ^ InvSbox(fc ^ k_guess) == gf_mult(scalar, e):
                                byte_cands[i].append(k_guess)
                                
                    # 3. If ALL 4 bytes have a solution for this specific 'e' and row guess
                    if all(len(cands) > 0 for cands in byte_cands):
                        # Link them together into 4-byte tuples and add to our valid set
                        for combo in itertools.product(*byte_cands):
                            valid_tuples_for_pair.add(combo)
                            
            if column_candidates[active_group_idx] is None:
                column_candidates[active_group_idx] = valid_tuples_for_pair
            else:
                # previous faults, need to use intersection to filter 
                column_candidates[active_group_idx] = column_candidates[active_group_idx].intersection(valid_tuples_for_pair)
        
        print(f"Processed all pairs in: {time.time() - start_time} seconds")
        
        final_round10_key = [0] * 16
        key_search_space = 1
        
        for group_idx in range(4):
            group_cands = column_candidates[group_idx]
            
            if group_cands is None:
                print(f"⚠️ WARNING: No faults collected for column {group_idx}. Completely unknown.")
                key_search_space *= (2**32)
            elif len(group_cands) == 0:
                print(f"❌ ERROR: Intersection resulted in 0 valid tuples for column {group_idx}.")
                key_search_space *= (2**32) # Treat as unknown space
            elif len(group_cands) > 1:
                key_search_space *= len(group_cands)
                print(f"⚠️ WARNING: Column {group_idx} still has {len(group_cands)} valid tuples. Need more faults!")
            else:
                # Exactly 1 tuple remains! Map the 4 bytes back to their original 1D indices
                correct_tuple = list(group_cands)[0]
                for i in range(4):
                    ci = all_target_groups[group_idx][i]
                    final_round10_key[ci] = correct_tuple[i]

        if key_search_space == 1:
            # we won!
            print("✅ Successfully recovered AES-128 key!")

        print(f"\n===> Recovered round 10 AES key: {bytes(final_round10_key).hex()}")
        
        master_key = self.recover_master_key_from_r10_key(final_round10_key)

        print(f"===> Key search space: {key_search_space} ~~ 2^{math.log2(key_search_space) if key_search_space > 0 else 0:.2f}")
        print(f"===> Recovered master AES key: {master_key}")
                    
        return master_key

In [325]:
d = DFASolver(pairs).solve()

Processed all pairs in: 13.511492729187012 seconds
✅ Successfully recovered AES-128 key!

===> Recovered round 10 AES key: 04adba95639f05073455767ff6c05c25
===> Key search space: 1 ~~ 2^0.00
===> Recovered master AES key: 65656373343735206565637334373573


# Dump key schedule for debugging

In [256]:
def keySchedule(KEY):
    ROUNDKEY = []
    hexKey = keyToHexArray(KEY)
    ROUNDKEY.append(hexKey)
    for i in range(0, 10):
        prev_arr = ROUNDKEY[-1]
        last_col = prev_arr[4-1]
        shift_col = arrayShift(last_col)
        sbox_col = arraySbox(shift_col)
        col_1 = xorArray(prev_arr[0], sbox_col, rcon=i)
        col_2 = xorArray(col_1, prev_arr[1])
        col_3 = xorArray(col_2, prev_arr[2])
        col_4 = xorArray(col_3, prev_arr[3])
        new_arr = np.array([col_1, col_2, col_3, col_4])
        ROUNDKEY.append(new_arr)
    return ROUNDKEY

In [257]:
def dumpKeySchedule():
    print("Key schedule:")
    for i in keySchedule(encoded_key):
        print(bytes(list(i.flatten())).hex())

In [258]:
print(dumpKeySchedule())

Key schedule:
65656373343735206565637334373573
fef3ec6bcac4d94bafa1ba389b968f4b
6c805f7fa644863409e53c0c9273b347
e7edff3041a97904484c4508da3ff64f
9aaf7b67db060263934a476b4975b124
17674d5ccc614f3f5f2b0854165eb970
6f311c1ba3505324fc7b5b70ea25e200
10a97f9cb3f92cb84f8277c8a5a795c8
cc83979a7f7abb2230f8ccea955f5922
184804b06732bf9257ca7378c2952a5a
04adba95639f05073455767ff6c05c25
None
